**1. Data Preparation**

**Step - 1: Load Dataset**

In [ ]:
import pandas as pd

ratings = pd.read_csv('/content/rating.csv', on_bad_lines='skip')
movies = pd.read_csv('/content/movie.csv', on_bad_lines='skip')

**Step - 2: Merge Datasets**

In [ ]:
data = pd.merge(ratings, movies, on='movieId')

**Step - 3: Check Data Info**

In [ ]:
print(data.info())
print(data.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 319682 entries, 0 to 319681
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     319682 non-null  int64  
 1   movieId    319682 non-null  int64  
 2   rating     319682 non-null  float64
 3   timestamp  319681 non-null  object 
 4   title      319682 non-null  object 
 5   genres     319682 non-null  object 
dtypes: float64(1), int64(2), object(3)
memory usage: 14.6+ MB
None
userId       0
movieId      0
rating       0
timestamp    1
title        0
genres       0
dtype: int64


**Step - 4: Convert rating into numeric**

In [ ]:
data['rating'] = pd.to_numeric(data['rating'], errors='coerce')

**Step - 5: Remove Null Values**

In [ ]:
data = data.dropna(subset=['rating'])

**Step - 6: Remove Duplicate Data**

In [ ]:
data = data.drop_duplicates()

**Step- 7: Remove Noisy Data**

In [ ]:
#Filter unpopular movies
movie_counts = data['title'].value_counts()
popular_movies = movie_counts[movie_counts >= 50].index

In [ ]:
#Filter inactive users
user_counts = data['userId'].value_counts()
active_users = user_counts[user_counts >= 50].index

In [ ]:
#Applying Filter
filtered_data = data[
    (data['title'].isin(popular_movies)) &
    (data['userId'].isin(active_users))
]

**Step - 8: Clean Dataset**

In [ ]:
print(filtered_data.shape)
filtered_data.head()

(216558, 6)


,userId,movieId,rating,timestamp,title,genres
0,1,2,3.5,2005-04-02 23:53:47,Jumanji (1995),Adventure|Children|Fantasy
1,1,29,3.5,2005-04-02 23:31:16,"City of Lost Children, The (Cité des enfants p...",Adventure|Drama|Fantasy|Mystery|Sci-Fi
2,1,32,3.5,2005-04-02 23:33:39,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller
3,1,47,3.5,2005-04-02 23:32:07,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,3.5,2005-04-02 23:29:40,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


**2.Feature Engineering**

In [ ]:
# create User-Movie Matrix
movie_matrix = filtered_data.pivot_table(index='userId', columns='title', values='rating')
movie_matrix = movie_matrix.fillna(0)

**3.Model Building**

In [ ]:
#corr_matrix = movie_matrix.corr(method='pearson')
corr_matrix = movie_matrix.corr(method='pearson')

**4. Recommendation System**

In [ ]:
def recommend(movie_name, n=5):
    matches = [m for m in corr_matrix.columns if movie_name.lower() in m.lower()]

    if len(matches) == 0:
        print("❌ Movie not found")
        return

    movie_name = matches[0]
    print(f"\n✅ Using: {movie_name}")

    similar_movies = corr_matrix[movie_name].dropna()
    similar_movies = similar_movies.sort_values(ascending=False)

    print(f"\n🎬 Top {n} recommendations:\n")
    print(similar_movies.iloc[1:n+1])

In [ ]:
while True:
    movie = input("\n🎬 Enter movie name (or type 'exit'): ")

    if movie.lower() == 'exit':
        print("👋 Exiting...")
        break

    recommend(movie)


🎬 Enter movie name (or type 'exit'): toy story

✅ Using: Toy Story (1995)

🎬 Top 5 recommendations:

title
Toy Story 2 (1999)           0.456102
Lion King, The (1994)        0.327667
Back to the Future (1985)    0.326422
Aladdin (1992)               0.314577
Bug's Life, A (1998)         0.314256
Name: Toy Story (1995), dtype: float64

🎬 Enter movie name (or type 'exit'): jumanji

✅ Using: Jumanji (1995)

🎬 Top 5 recommendations:

title
Mask, The (1994)            0.428498
Home Alone (1990)           0.383059
Santa Clause, The (1994)    0.380677
Lion King, The (1994)       0.380627
Casper (1995)               0.369032
Name: Jumanji (1995), dtype: float64

🎬 Enter movie name (or type 'exit'): seven

✅ Using: Magnificent Seven, The (1960)

🎬 Top 5 recommendations:

title
For a Few Dollars More (Per qualche dollaro in più) (1965)    0.353438
Dirty Dozen, The (1967)                                       0.302836
Dirty Harry (1971)                                            0.301902
Fistful